# Staged Data QA
**BRIM Systems** | Prepared by: Brian Rimmer

---

## Purpose

Validation and exploration pass against dbt staging model output. Confirms that staging
transformations resolved the issues identified in `raw_profiling.ipynb`, and surfaces
any remaining issues before intermediate models are built.

**Section flow:**
1. Configuration & Data Loading
2. Staged Data Summary (head preview + schema & null flags)
3. Known-Value Validation
4. Exploration & Visualization (numeric, categorical, correlation, time series)
5. Structural Integrity Checks (PK, RI & join viability, FK null rates)
6. Additional Analyses (numeric range, temporal gap detection)

**Sequencing:**
```
raw_profiling.ipynb  ->  dbt staging models  ->  staged_qa.ipynb  (this file)
        ->  int_analysis.ipynb  ->  dbt intermediate + mart models
```

Run after `dbt run --select staging` has completed successfully.

---

> **Note for reviewers:** This is a read-only validation artifact. No files are written to disk.

## 0. Environment Setup

In [1]:
import warnings
from pathlib import Path

import duckdb
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from scipy.stats import linregress

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 60)
pd.set_option("display.max_colwidth", 80)
pd.set_option("display.float_format", "{:,.4f}".format)

BRAND_BLUE   = "#3D5166"
BRAND_ACCENT = "#6B8FA8"
WARN_AMBER   = "#D4881E"
FAIL_RED     = "#B94040"

plt.rcParams.update({
    "figure.facecolor": "white", "axes.facecolor": "white",
    "axes.edgecolor": "#CCCCCC", "axes.grid": True,
    "grid.color": "#EEEEEE", "grid.linestyle": "-",
    "font.family": "sans-serif", "font.size": 11,
    "axes.titlesize": 13, "axes.titleweight": "bold",
    "axes.labelsize": 11, "xtick.labelsize": 10,
    "ytick.labelsize": 10, "legend.fontsize": 10,
    "figure.dpi": 120,
})

print("Environment ready.")

Environment ready.


## 1. Configuration & Data Loading

Universal configuration shared across all sections. Variables specific to an
individual analysis are defined at the top of that section.

**Shared with `raw_profiling.ipynb`** — keep null thresholds in sync.
**What to update for a new project** — `DB_PATH`, all `TBL_STG_*` constants,
and all config dicts below.

In [2]:
# ── Shared with raw_profiling.ipynb — keep in sync ────────────────────────
TBL_MACHINES          = "machines"
TBL_OPERATORS         = "operators"
TBL_MATERIAL_LOTS     = "material_lots"
TBL_PART_CATALOG      = "part_catalog"
TBL_PRODUCTION_ORDERS = "production_orders"
TBL_INSPECTION_REC    = "inspection_records"
TBL_SCRAP_EVENTS      = "scrap_events"


# ── Database connection ────────────────────────────────────────────────────
DB_PATH = Path("../data_source/defects_scrap.duckdb").resolve()

# ── Staged table name constants ───────────────────────────────────────────
TBL_STG_MACHINES  = "stg_mes__machines"
TBL_STG_OPERATORS = "stg_hr__operators"
TBL_STG_LOTS      = "stg_materials__lots"
TBL_STG_PARTS     = "stg_erp__part_catalog"
TBL_STG_ORDERS    = "stg_erp__production_orders"
TBL_STG_INSP      = "stg_qms__inspection_records"
TBL_STG_SCRAP     = "stg_qms__scrap_events"

TABLES = {
    TBL_STG_MACHINES:  TBL_STG_MACHINES,
    TBL_STG_OPERATORS: TBL_STG_OPERATORS,
    TBL_STG_LOTS:      TBL_STG_LOTS,
    TBL_STG_PARTS:     TBL_STG_PARTS,
    TBL_STG_ORDERS:    TBL_STG_ORDERS,
    TBL_STG_INSP:      TBL_STG_INSP,
    TBL_STG_SCRAP:     TBL_STG_SCRAP,
}

# ── Primary keys ───────────────────────────────────────────────────────────
PK_MAP = {
    TBL_STG_MACHINES:  "machine_id",
    TBL_STG_OPERATORS: "operator_id",
    TBL_STG_LOTS:      "lot_id",
    TBL_STG_PARTS:     "part_number",
    TBL_STG_ORDERS:    "work_order_id",
    TBL_STG_INSP:      "inspection_id",
    TBL_STG_SCRAP:     "scrap_id",
}

# ── Foreign key relationships ─────────────────────────────────────────────
# (child_table, child_col, parent_table, parent_col)
FK_RELATIONSHIPS = [
    (TBL_STG_ORDERS, "machine_id",    TBL_STG_MACHINES,  "machine_id"),
    (TBL_STG_ORDERS, "operator_id",   TBL_STG_OPERATORS, "operator_id"),
    (TBL_STG_ORDERS, "part_number",   TBL_STG_PARTS,     "part_number"),
    (TBL_STG_ORDERS, "lot_id",        TBL_STG_LOTS,      "lot_id"),
    (TBL_STG_INSP,   "work_order_id", TBL_STG_ORDERS,    "work_order_id"),
    (TBL_STG_SCRAP,  "work_order_id", TBL_STG_ORDERS,    "work_order_id"),
    (TBL_STG_SCRAP,  "inspection_id", TBL_STG_INSP,      "inspection_id"),
]

# ── FK columns expected to be partially null (structural nulls) ───────────
FK_NULLABLE = {
    (TBL_STG_ORDERS, "lot_id"),  # material not always scanned at job start
}


# ── Load staged tables from DuckDB ────────────────────────────────────────
dfs = {}

if not DB_PATH.exists():
    print(f"  ✗  DuckDB file not found: {DB_PATH}")
    print("     Run: dbt run --select staging")
else:
    con = duckdb.connect(str(DB_PATH), read_only=True)
    print(f"{'Table':<38} {'Rows':>8}  {'Cols':>5}  Status")
    print("-" * 65)
    for name, view_name in TABLES.items():
        try:
            df = con.execute(f"SELECT * FROM main.{view_name}").df()
            dfs[name] = df
            print(f"  ✓  {name:<35} {len(df):>8,}  {len(df.columns):>5}")
        except Exception as e:
            print(f"  ✗  {name:<35} ERROR: {e}")
    con.close()

orders = dfs.get(TBL_STG_ORDERS)
insp   = dfs.get(TBL_STG_INSP)
scrap  = dfs.get(TBL_STG_SCRAP)
mach   = dfs.get(TBL_STG_MACHINES)
ops    = dfs.get(TBL_STG_OPERATORS)
lots   = dfs.get(TBL_STG_LOTS)
parts  = dfs.get(TBL_STG_PARTS)

  ✗  DuckDB file not found: /workspaces/portfolio/defects_scrap/etl_pipeline/data/defects_scrap.duckdb
     Run: dbt run --select staging


---
## 2. Staged Data Summary

### 2.1 Preview

In [3]:
for tbl, df in dfs.items():
    full_dups = df.duplicated(keep=False).sum()
    dup_str   = f"{full_dups:,}" if full_dups > 0 else "none"
    print(f"\n{'='*70}")
    print(f"  {tbl.upper()}  ({len(df):,} rows  x  {len(df.columns)} columns)  |  Full duplicates: {dup_str}")
    print(f"{'='*70}")
    display(df.head())

### 2.2 Schema & Null Flags

In [4]:
# NULL THRESHOLDS
NULL_WARN_PCT = 0
NULL_FAIL_PCT = 10.0

In [5]:
def schema_summary(df):
    rows = []
    for col in df.columns:
        s        = df[col]
        null_ct  = s.isna().sum()
        null_pct = null_ct / len(df) * 100
        rows.append({
            "column":     col,
            "dtype":      str(s.dtype),
            "null_count": null_ct,
            "null_pct":   round(null_pct, 1),
            "n_unique":   s.nunique(dropna=True),
            "sample":     str(s.dropna().iloc[0])[:60] if null_ct < len(df) else "(all null)",
        })
    return pd.DataFrame(rows)

for tbl, df in dfs.items():
    print(f"\n{'='*70}\n  {tbl.upper()}  ({len(df):,} rows, {len(df.columns)} columns)\n{'='*70}")
    display(
        schema_summary(df).style
        .format({"null_pct": "{:.1f}%"})
        .applymap(
            lambda v: "background-color: #F8D7DA" if isinstance(v, float) and v > NULL_FAIL_PCT else
                      "background-color: #FFF3CD" if isinstance(v, float) and v > NULL_WARN_PCT else "",
            subset=["null_pct"]
        )
        .set_properties(**{"font-size": "11px"})
    )

---
## 3. Exploration & Visualization

In [6]:
# Auto-classify columns into numeric and categorical.
# Numeric: includes all int, float, complex dtypes ("number" is a pandas alias).
# Categorical: object and pandas Categorical dtypes, limited to <= 30 distinct values.
NUMERIC_COLS     = {}
CATEGORICAL_COLS = {}

for tbl, df in dfs.items():
    num_cols = df.select_dtypes(include=["number"]).columns.tolist()
    cat_cols = [
        c for c in df.select_dtypes(include=["object", "category"]).columns
        if df[c].nunique() <= 30
    ]
    if num_cols:
        NUMERIC_COLS[tbl]     = num_cols
    if cat_cols:
        CATEGORICAL_COLS[tbl] = cat_cols

print("Column classification:")
for tbl in dfs:
    n = len(NUMERIC_COLS.get(tbl, []))
    c = len(CATEGORICAL_COLS.get(tbl, []))
    print(f"  {tbl:<38} numeric: {n:>3}   categorical: {c:>3}")

Column classification:


### 3.1 Numeric Distributions

In [7]:
# Histogram + box plot per numeric column per table.
for tbl, cols in NUMERIC_COLS.items():
    df = dfs[tbl]
    n  = len(cols)
    if n == 0:
        continue
    fig, axes = plt.subplots(2, n, figsize=(max(4 * n, 8), 7))
    if n == 1:
        axes = np.array([[axes[0]], [axes[1]]])

    for i, col in enumerate(cols):
        s = df[col].dropna()
        axes[0, i].hist(s, bins=40, color=BRAND_BLUE, edgecolor="white", linewidth=0.3)
        axes[0, i].set_title(col, fontsize=10)
        axes[0, i].set_ylabel("Count" if i == 0 else "")
        axes[0, i].xaxis.set_major_formatter(
            mticker.FuncFormatter(lambda x, _: f"{x:,.0f}")
        )
        axes[1, i].boxplot(s, vert=True, patch_artist=True,
                           boxprops=dict(facecolor=BRAND_ACCENT, color=BRAND_BLUE),
                           medianprops=dict(color=WARN_AMBER, linewidth=2),
                           flierprops=dict(marker="o", markersize=3,
                                          markerfacecolor=FAIL_RED, alpha=0.5))
        axes[1, i].set_title(f"{col} (box)", fontsize=10)

    fig.suptitle(f"{tbl} — Numeric Distributions", fontsize=13, fontweight="bold")
    plt.tight_layout()
    plt.show()

### 3.2 Categorical Distributions

In [8]:
# Bar chart of value frequencies per categorical column per table.
for tbl, cols in CATEGORICAL_COLS.items():
    df = dfs[tbl]
    n  = len(cols)
    if n == 0:
        continue
    ncols_plot = min(n, 3)
    nrows_plot = (n + ncols_plot - 1) // ncols_plot
    fig, axes  = plt.subplots(nrows_plot, ncols_plot,
                               figsize=(6 * ncols_plot, 4 * nrows_plot))
    axes = np.array(axes).flatten()

    for i, col in enumerate(cols):
        vc = df[col].fillna("(null)").value_counts()
        axes[i].bar(range(len(vc)), vc.values, color=BRAND_BLUE, width=0.7)
        axes[i].set_xticks(range(len(vc)))
        axes[i].set_xticklabels(vc.index, rotation=35, ha="right", fontsize=9)
        axes[i].set_title(col, fontsize=10)
        axes[i].set_ylabel("Count")
        axes[i].yaxis.set_major_formatter(
            mticker.FuncFormatter(lambda x, _: f"{x:,.0f}")
        )

    for j in range(i + 1, len(axes)):
        axes[j].set_visible(False)

    fig.suptitle(f"{tbl} — Categorical Distributions", fontsize=13, fontweight="bold")
    plt.tight_layout()
    plt.show()

### 3.3 Correlation Matrices

In [9]:
# Pearson correlation heatmap for tables with 3+ numeric columns.
for tbl, cols in NUMERIC_COLS.items():
    if len(cols) < 3:
        continue
    corr = dfs[tbl][cols].corr()
    fig, ax = plt.subplots(figsize=(max(6, len(cols) * 1.2), max(5, len(cols))))
    sns.heatmap(
        corr, ax=ax, annot=True, fmt=".2f",
        cmap="coolwarm", center=0,
        linewidths=0.5, linecolor="white",
        vmin=-1, vmax=1
    )
    ax.set_title(f"{tbl} — Numeric Correlation Matrix")
    plt.tight_layout()
    plt.show()

### 3.4 Time Series Analysis

Line chart with trend overlay and summary statistics for each table with a date column.

| Statistic | What it tells you |
|-----------|-------------------|
| Mean / Median | Central tendency; median is robust to spikes |
| Std dev + CV | Spread; CV (std/mean) enables cross-series volatility comparison |
| Min / Max (with dates) | Extremes and when they occurred |
| Trend slope | Direction and rate of change (linear regression, records/month) |
| Lag-1 autocorrelation | Does this month predict the next? High = momentum, low = noise |
| Seasonal strength | Consistency of within-year patterns across years (requires 2+ years) |

In [10]:
# Date columns — moved here from Section 1 as they are specific to this analysis.
DATE_COLUMNS = {
    TBL_STG_ORDERS:    ["order_date", "actual_start"],
    TBL_STG_INSP:      ["inspection_date"],
    TBL_STG_SCRAP:     ["scrap_date"],
    TBL_STG_LOTS:      ["receipt_date"],
    TBL_STG_OPERATORS: ["hire_date"],
}

for tbl, date_cols in DATE_COLUMNS.items():
    if tbl not in dfs:
        continue
    df       = dfs[tbl]
    date_col = next((c for c in date_cols if c in df.columns), None)
    if date_col is None:
        continue

    tmp           = df.copy()
    tmp["_dt"]    = pd.to_datetime(tmp[date_col], errors="coerce")
    tmp["_month"] = tmp["_dt"].dt.to_period("M")
    monthly       = tmp.groupby("_month").size().reset_index(name="records")
    monthly["ym"] = monthly["_month"].astype(str)
    s             = monthly["records"]

    if len(s) < 3:
        continue

    # ── Line chart ────────────────────────────────────────────────────────
    x_num                      = np.arange(len(s))
    slope, intercept, _, _, _  = linregress(x_num, s)
    trend                      = slope * x_num + intercept

    fig, ax = plt.subplots(figsize=(14, 4))
    ax.plot(monthly["ym"], s, color=BRAND_BLUE, linewidth=2, marker="o", markersize=4)
    ax.axhline(s.mean(), color=WARN_AMBER, linestyle="--", linewidth=1.2,
               label=f"Mean ({s.mean():,.0f})")
    ax.plot(monthly["ym"], trend, color=FAIL_RED, linewidth=1.5,
            linestyle="-.", label=f"Trend (slope: {slope:+.1f}/mo)")
    ax.set_title(f"Time Series Analysis — {tbl} ({date_col})")
    ax.set_ylabel("Records")
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:,.0f}"))
    xticks = monthly["ym"][::6]
    ax.set_xticks(xticks)
    ax.set_xticklabels(xticks, rotation=45, ha="right")
    ax.legend()
    plt.tight_layout()
    plt.show()

    # ── Summary statistics ────────────────────────────────────────────────
    cv      = s.std() / s.mean() if s.mean() != 0 else None
    acf1    = s.autocorr(lag=1) if len(s) > 1 else None
    min_idx = s.idxmin()
    max_idx = s.idxmax()

    if len(monthly) >= 24:
        monthly["_moy"] = [p.month for p in monthly["_month"]]
        moy_means       = monthly.groupby("_moy")["records"].mean()
        between_var     = moy_means.var()
        total_var       = s.var()
        seasonal_str    = round(between_var / total_var, 3) if total_var > 0 else None
    else:
        seasonal_str = None

    print(f"  {tbl} — {date_col}")
    print(f"  {'─'*55}")
    print(f"  {'Mean:':<30} {s.mean():>10,.1f}")
    print(f"  {'Median:':<30} {s.median():>10,.1f}")
    print(f"  {'Std dev:':<30} {s.std():>10,.1f}")
    if cv is not None:
        print(f"  {'CV (std/mean):':<30} {cv:>10.3f}")
    print(f"  {'Min:':<30} {s.min():>10,.0f}  ({monthly.loc[min_idx, 'ym']})")
    print(f"  {'Max:':<30} {s.max():>10,.0f}  ({monthly.loc[max_idx, 'ym']})")
    print(f"  {'Trend slope (records/mo):':<30} {slope:>+10.2f}")
    if acf1 is not None:
        print(f"  {'Lag-1 autocorrelation:':<30} {acf1:>10.3f}")
    if seasonal_str is not None:
        label = 'strong' if seasonal_str > 0.3 else 'weak'
        print(f"  {'Seasonal strength:':<30} {seasonal_str:>10.3f}  ({label})")
    else:
        print(f"  {'Seasonal strength:':<30} {'< 2 years of data':>10}")
    print()

---
## 4. Structural Integrity Checks

### 4.1 Primary Key Validation

Staging models should produce fully deduplicated tables. Any PK violation means
staging deduplication logic is incomplete and must be fixed before mart models are built.

In [11]:
# Thresholds
JOIN_WARN_PCT = 100.0  # flag any join below 100% match rate

In [12]:
pk_flags = []

print(f"{'Table':<38} {'PK Column':<22} {'Total':>8}  {'Unique':>8}  "
      f"{'Dupes':>8}  {'Dup %':>7}  Status")
print("-" * 100)

for tbl, pk in PK_MAP.items():
    if tbl not in dfs:
        continue
    df = dfs[tbl]
    if pk not in df.columns:
        print(f"  ✗  {tbl:<35} {pk:<22} COLUMN MISSING")
        continue
    total   = len(df)
    unique  = df[pk].nunique()
    dupes   = total - unique
    dup_pct = dupes / total * 100
    status  = "✓ PASS" if dupes == 0 else "✗ FAIL"
    print(f"  {status[:1]}  {tbl:<35} {pk:<22} {total:>8,}  "
          f"{unique:>8,}  {dupes:>8,}  {dup_pct:>6.1f}%  {status}")
    if dupes > 0:
        pk_flags.append({
            "table": tbl, "column": pk,
            "issue": f"{dupes:,} duplicate rows ({dup_pct:.1f}%) after staging dedup"
        })

if pk_flags:
    print(f"\n✗ {len(pk_flags)} table(s) still have duplicate PKs. "
          "Fix dedup logic in staging models before proceeding.")

Table                                  PK Column                 Total    Unique     Dupes    Dup %  Status
----------------------------------------------------------------------------------------------------


### 4.2 Referential Integrity & Join Viability

Tests FK relationships defined in `FK_RELATIONSHIPS`. Any join below 100% match
rate is flagged. Two metrics reported per relationship:
- **Distinct match rate** — % of unique FK values that resolve to a valid PK
- **Row-level match rate** — % of rows that survive a left join to the parent

In [13]:
ri_flags = []

def ri_check(child_tbl, child_col, parent_tbl, parent_col):
    child_df  = dfs.get(child_tbl)
    parent_df = dfs.get(parent_tbl)
    if child_df is None or parent_df is None:
        print(f"  ✗  SKIP  {child_tbl} or {parent_tbl} not loaded")
        return
    if child_col not in child_df.columns or parent_col not in parent_df.columns:
        print(f"  ✗  SKIP  Column missing in {child_tbl}.{child_col}")
        return

    child_vals     = set(child_df[child_col].dropna().unique())
    parent_vals    = set(parent_df[parent_col].dropna().unique())
    orphans        = child_vals - parent_vals
    distinct_match = (len(child_vals) - len(orphans)) / max(len(child_vals), 1) * 100

    merged    = child_df.merge(
        parent_df[[parent_col]].drop_duplicates(),
        left_on=child_col, right_on=parent_col, how="left", indicator=True
    )
    row_match = (merged["_merge"] == "both").mean() * 100
    status    = "✓ PASS" if row_match >= JOIN_WARN_PCT else "✗ FAIL"

    print(f"  {status}  {child_tbl}.{child_col} -> {parent_tbl}.{parent_col}")
    print(f"         Row match: {row_match:.1f}%   "
          f"Distinct match: {distinct_match:.1f}%   "
          f"Orphan values: {len(orphans):,}")
    if orphans:
        print(f"         Sample orphan values: {list(orphans)[:6]}")
    print()

    if row_match < JOIN_WARN_PCT:
        ri_flags.append({
            "table":  child_tbl, "column": child_col,
            "issue":  f"Row match {row_match:.1f}% -> {parent_tbl}.{parent_col} "
                      f"({len(orphans):,} orphan values)"
        })

print("REFERENTIAL INTEGRITY & JOIN VIABILITY")
print("=" * 70)
for child_tbl, child_col, parent_tbl, parent_col in FK_RELATIONSHIPS:
    ri_check(child_tbl, child_col, parent_tbl, parent_col)

REFERENTIAL INTEGRITY & JOIN VIABILITY
  ✗  SKIP  stg_erp__production_orders or stg_mes__machines not loaded
  ✗  SKIP  stg_erp__production_orders or stg_hr__operators not loaded
  ✗  SKIP  stg_erp__production_orders or stg_erp__part_catalog not loaded
  ✗  SKIP  stg_erp__production_orders or stg_materials__lots not loaded
  ✗  SKIP  stg_qms__inspection_records or stg_erp__production_orders not loaded
  ✗  SKIP  stg_qms__scrap_events or stg_erp__production_orders not loaded
  ✗  SKIP  stg_qms__scrap_events or stg_qms__inspection_records not loaded


### 4.3 FK Null Rates

FK columns should be fully populated unless explicitly declared as structural nulls
in `FK_NULLABLE`. Unexpected nulls indicate a staging join or derivation issue.

In [14]:
fk_null_flags = []

print(f"{'Relationship':<65} {'Null %':>8}  {'Structural':>12}  Status")
print("-" * 100)

for child_tbl, child_col, parent_tbl, parent_col in FK_RELATIONSHIPS:
    if child_tbl not in dfs or child_col not in dfs[child_tbl].columns:
        continue
    df          = dfs[child_tbl]
    null_pct    = df[child_col].isna().mean() * 100
    is_nullable = (child_tbl, child_col) in FK_NULLABLE
    status      = (
        "✓ PASS"   if null_pct == 0 else
        "✓ STRUCT" if is_nullable else
        "✗ FAIL"
    )
    rel = f"{child_tbl}.{child_col} -> {parent_tbl}.{parent_col}"
    print(f"  {status[:1]}  {rel:<63} {null_pct:>7.1f}%  "
          f"{'yes' if is_nullable else 'no':>12}  {status}")
    if null_pct > 0 and not is_nullable:
        fk_null_flags.append({
            "table":  child_tbl, "column": child_col,
            "issue":  f"FK null rate {null_pct:.1f}% — not declared in FK_NULLABLE"
        })

Relationship                                                        Null %    Structural  Status
----------------------------------------------------------------------------------------------------


---
## 5. Additional Analyses

### 5.1 Numeric Range Validation

Confirms numeric columns fall within expected bounds after staging type casts.
Bounds defined in `NUMERIC_BOUNDS` (Section 1). Out-of-range values indicate
a data entry error in the source system or an unexpected staging cast result.

In [15]:
# ── Numeric bounds ───────────────────────────────
NUMERIC_BOUNDS = {
    TBL_STG_ORDERS: {
        "quantity_ordered": (1, None),
        "std_labor_hrs":    (0, None),
    },
    TBL_STG_INSP: {
        "quantity_inspected": (0, None),
        "quantity_passed":    (0, None),
        "quantity_failed":    (0, None),
    },
    TBL_STG_SCRAP: {
        "quantity_scrapped":      (0, None),
        "quantity_reworked":      (0, None),
        "material_cost_per_unit": (0, None),
        "labor_cost_per_unit":    (0, None),
        "total_scrap_cost":       (0, None),
    },
    TBL_STG_LOTS: {
        "quantity_lbs":     (1, None),
        "unit_cost_per_lb": (0, None),
    },
    TBL_STG_MACHINES: {
        "age_years": (0, 50),
    },
}
range_flags = []

print(f"{'Table':<38} {'Column':<25} {'Min':>10}  {'Max':>10}  "
      f"{'Violations':>12}  {'Viol %':>8}  Status")
print("-" * 115)

for tbl, col_bounds in NUMERIC_BOUNDS.items():
    if tbl not in dfs:
        continue
    df = dfs[tbl]
    for col, (lo, hi) in col_bounds.items():
        if col not in df.columns:
            print(f"  ✗  {tbl:<35} {col:<25} COLUMN MISSING")
            continue
        s         = df[col].dropna()
        lo_viol   = (s < lo).sum() if lo is not None else 0
        hi_viol   = (s > hi).sum() if hi is not None else 0
        viols     = lo_viol + hi_viol
        viol_pct  = viols / len(df) * 100
        status    = "✓ PASS" if viols == 0 else "✗ FAIL"
        bound_str = f"[{lo if lo is not None else '-inf'}, {hi if hi is not None else 'inf'}]"
        print(f"  {status[:1]}  {tbl:<35} {col:<25} "
              f"{s.min():>10,.2f}  {s.max():>10,.2f}  "
              f"{viols:>12,}  {viol_pct:>7.2f}%  {status}")
        if viols > 0:
            range_flags.append({
                "table":  tbl, "column": col,
                "issue":  f"{viols:,} values outside bounds {bound_str} "
                          f"(below: {lo_viol:,}, above: {hi_viol:,})"
            })

Table                                  Column                           Min         Max    Violations    Viol %  Status
-------------------------------------------------------------------------------------------------------------------


### 5.2 Temporal Gap Detection

Checks for missing periods in time series data. A gap is defined as a calendar
month with zero records that falls within the overall date range of the table.
Uses the same `DATE_COLUMNS` config defined in Section 4.4.

In [16]:
temporal_flags = []

for tbl, date_cols in DATE_COLUMNS.items():
    if tbl not in dfs:
        continue
    df       = dfs[tbl]
    date_col = next((c for c in date_cols if c in df.columns), None)
    if date_col is None:
        continue

    parsed = pd.to_datetime(df[date_col], errors="coerce")
    if parsed.isna().all():
        continue

    periods  = parsed.dt.to_period("M").dropna()
    min_per  = periods.min()
    max_per  = periods.max()
    expected = pd.period_range(min_per, max_per, freq="M")
    observed = set(periods.unique())
    missing  = [p for p in expected if p not in observed]

    if missing:
        print(f"  ⚠  {tbl}.{date_col}  —  {len(missing)} missing month(s) "
              f"in range {min_per} to {max_per}:")
        for p in missing:
            print(f"       {p}")
        temporal_flags.append({
            "table":  tbl, "column": date_col,
            "issue":  f"{len(missing)} missing month(s): "
                      f"{[str(p) for p in missing[:5]]}"
        })
    else:
        print(f"  ✓  {tbl}.{date_col}  —  no gaps detected "
              f"({min_per} to {max_per}, {len(expected)} months)")